# 02 — Hyperparameter Tuning (Optuna)

**Pipeline stage 2 of 7.** Optuna study over the LSTM hyperparameter space.
This is the most expensive notebook in the pipeline; isolate it so you only re-run
when the data pipeline (notebook 01) actually changes.

**Inputs**
- Sequences from notebook 01 (loaded from `output_refactored/`)
- `shared_config.py`

**Outputs**
- `best_hps.json` — winning hyperparameter dictionary (consumed by notebook 03)
- Optuna study artifacts and diagnostic plots in `OUTPUT_DIR`

> Source: Parts 8–9 of the original monolithic notebook.


## Setup

In [1]:
!pip install -q optuna optuna-integration plotly kaleido scikit-learn tensorflow joblib


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (average_precision_score, precision_recall_curve)

import optuna
from optuna.samplers import TPESampler

from shared_config import *
print_config_summary()

--- Workflow Configuration Summary ---
Enhanced Features Enabled:     True
Scaler Type Selected:          Robust
Class Weighting Enabled:       True
Hyperparameter Tuning Enabled: True
EnKF Enabled:                  True
EnKF Propagator:               SeasonalAR1
FAST_TEST Mode:                False
Sequence Length (default):     26
Forecast Horizon:              1 week(s)
Output directory:              output_refactored/
--------------------------------------


C:\Users\victo\PycharmProjects\PythonProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load upstream artifacts (sequences from notebook 01)

In [3]:
from lstm_utils import (create_sequences_from_df, build_lstm_model,
                        class_weights_powered, build_tunable_lstm)
from pipeline_utils import build_sequence_artifact, load_sequence_artifact, HPsShim, write_json

# Notebook 01 writes both branches. Notebook 02 will tune them under the same
# Optuna budget and then persist tuned sequence artifacts for notebook 03.
raw_seq = load_sequence_artifact(RAW_SEQUENCES_PATH)
enkf_seq = load_sequence_artifact(ENKF_SEQUENCES_PATH)

X_train = raw_seq["X_train"]; y_train = raw_seq["y_train"]
X_val = raw_seq["X_val"]; y_val = raw_seq["y_val"]
X_test = raw_seq["X_test"]; y_test = raw_seq["y_test"]
feature_columns = raw_seq["feature_columns"]

print(f"Raw X_train:  {raw_seq['X_train'].shape}, y_train: {raw_seq['y_train'].shape}")
print(f"EnKF X_train: {enkf_seq['X_train'].shape}, y_train: {enkf_seq['y_train'].shape}")


Raw X_train:  (1097, 26, 52), y_train: (1097,)
EnKF X_train: (1097, 26, 58), y_train: (1097,)



## Part 8: Model Definition (LSTM).

This part defines the function build_lstm_model responsible for creating the LSTM network architecture using TensorFlow/Keras. The function is designed to be flexible: it can use default hyperparameters defined in Part 1, or it can accept a hp object from KerasTuner (which we'll use later in Part 8) to create models with varying hyperparameters during the tuning process.

This cell defines the build_lstm_model function. It specifies the layers (LSTM, Dropout, Dense) and allows for hyperparameter configuration either through defaults or a KerasTuner object. It then demonstrates building the model with default hyperparameters, assuming the input shape is known from Part 6.

Explanation:

1. Function build_lstm_model:
    - Takes input_shape (required) and an optional KerasTuner hp object.
    - Hyperparameter Handling: If hp is provided (during tuning), it defines hyperparameters using hp.Int, hp.Float, hp.Choice. If hp is None (when building the default or final model), it uses the DEFAULT_ variables defined in Part 1 (with fallbacks just in case).
    - Architecture: Defines a two-layer LSTM structure with Dropout. You can easily modify this (e.g., change to GRU, add Dense layers, use Bidirectional) by editing this function.
    - Compilation: Compiles the model inside the function using Adam optimizer and binary cross-entropy loss. This is convenient for KerasTuner.

2. Main Execution (if __name__ == "__main__":):
    - Checks if X_train exists (from Part 6) to get the required input_shape.
    - Calls build_lstm_model with hp=None to create an instance using the default hyperparameters.
    - Prints the model summary.

After running this cell, the function build_lstm_model is defined and ready to be used either by KerasTuner (Part 8) or directly for training the default/final model (Part 9). The lstm_model_default variable holds an example compiled model instance (useful for checking).

In [4]:
# --- Test Block ---
if __name__ == "__main__":
    print("--- Testing Model Definition (Step 7) ---")
    if 'X_train' in locals() and X_train is not None:
        input_shape_test = (X_train.shape[1], X_train.shape[2])
        print(f"Input Shape detected: {input_shape_test}")
        try:
            model_test = build_lstm_model(input_shape_test, hps=None)
            print("\nModel built successfully!")
            model_test.summary()
        except Exception as e:
            print(f"Error building model: {e}")
    else:
        print("Warning: X_train not found. Run Step 6 first to test this function.")
        print("Function 'build_lstm_model' is defined but not tested.")


--- Testing Model Definition (Step 7) ---
Input Shape detected: (26, 52)

Model built successfully!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 26, 64)         │        29,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 26, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,401 (165.63 KB)

 Trainable params: 42,401 (165.63 KB)

 Non-trainable params: 0 (0.00 B)

## Part 9: Hyperparameter Tuning with KerasTuner (Conditional)

This cell sets up and runs the KerasTuner search process if the PERFORM_TUNING flag (from Part 1) is set to True. It uses the build_lstm_model function (from Part 7) as the hypermodel builder and searches for the best combination of LSTM units, dropout rate, and learning rate based on validation accuracy.

Explanation:

1. Conditional Execution: The entire cell's logic is wrapped in if can_tune:, which checks if PERFORM_TUNING was set to True in Part 1 and if the keras_tuner library (kt) was successfully imported. If not, it prints a message and skips tuning.
2. Prerequisite Checks: Inside the if block, it checks if the necessary sequence data (X_train, y_train, etc.) and the build_lstm_model function exist before proceeding.
3. Tuner Setup:
    - Creates a keras_tuner.RandomSearch instance (you could switch to kt.Hyperband for potentially faster convergence).
    - Passes a lambda function lambda hp: build_lstm_model(input_shape_tune, hp=hp) as the hypermodel builder.
    - This ensures the build_lstm_model function receives the tuner's hp object to define the model architecture with tunable parameters.
    - Sets the objective to 'val_accuracy' (tune for best accuracy on the validation set).
    - Configures max_trials, directory, project_name, etc.
4. Run Search:
    - Calls tuner.search(), passing the training and validation data.
    - Uses a dedicated EarlyStopping callback with potentially shorter patience for the tuning phase itself.
5. Retrieve Best Hyperparameters:
    - After the search completes, tuner.get_best_hyperparameters(num_trials=1)[0] retrieves the HyperParameters object corresponding to the best trial.
    - The values for the tuned hyperparameters (e.g., lstm_units_1, dropout_rate, learning_rate) are extracted and printed.

The best_hps variable stores this object for use in the next step (Part 9).
Includes error handling in case the tuner fails or doesn't return results.
If PERFORM_TUNING is False, this cell will simply print a message and set best_hps to None. The next step (Part 9) will then know to use the default hyperparameters defined in Part 1.

## Tuning Configuration

Adjust `N_TRIALS` and `MAX_TIME_HOURS` based on your compute budget. For initial exploration, 60 trials × 3 folds × 50 epochs (with pruning) typically completes in 4–8 hours on a single GPU. The SQLite store lets you stop and resume — re-running this cell will pick up where you left off.


In [5]:
# --- Tuning budget (FAST_TEST collapses everything to a few-minute smoke test) ---
if FAST_TEST:
    N_TRIALS         = 3
    MAX_TIME_HOURS   = 0.5
    N_CV_FOLDS       = 2
    N_STARTUP_TRIALS = 1
    MAX_EPOCHS       = 6
    MIN_EPOCHS       = 2
    EARLY_STOP_PAT   = 3
else:
    N_TRIALS         = 60
    MAX_TIME_HOURS   = 8
    N_CV_FOLDS       = 3
    N_STARTUP_TRIALS = 15
    MAX_EPOCHS       = 50
    MIN_EPOCHS       = 5
    EARLY_STOP_PAT   = 8
STABILITY_LAMBDA = 0.3

# --- Persistence ---
STUDY_NAME   = 'hab_lstm_v3_fast' if FAST_TEST else 'hab_lstm_v3'
STORAGE_PATH = f'sqlite:///{OUTPUT_DIR}/optuna_{STUDY_NAME}.db'
RESULTS_DIR  = os.path.join(OUTPUT_DIR, 'tuning_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Study name:    {STUDY_NAME}')
print(f'Storage:       {STORAGE_PATH}')
print(f'Trial budget:  {N_TRIALS} trials  /  {MAX_TIME_HOURS} h wall-clock')
print(f'Inside trial:  {N_CV_FOLDS}-fold TSCV, up to {MAX_EPOCHS} epochs each')
print(f'FAST_TEST:     {FAST_TEST}')


Study name:    hab_lstm_v3
Storage:       sqlite:///output_refactored//optuna_hab_lstm_v3.db
Trial budget:  60 trials  /  8 h wall-clock
Inside trial:  3-fold TSCV, up to 50 epochs each
FAST_TEST:     False


## Prepare the Tuning Pool

Optuna will need to rebuild sequences at every trial because `seq_length` is itself a hyperparameter. We combine `train_df` and `val_df` into a single chronological pool and let `TimeSeriesSplit` carve it up into expanding-window folds.

This keeps the held-out test set (`test_df`) completely untouched until final evaluation — same as the original pipeline.


In [6]:
# Restore notebook 01 artifacts for both tuning branches.
train_df = pd.read_parquet(TRAIN_DF_PATH)
validation_df = pd.read_parquet(VAL_DF_PATH)
test_df = pd.read_parquet(TEST_DF_PATH)
train_scaled_df = pd.read_parquet(TRAIN_SCALED_PATH)
validation_scaled_df = pd.read_parquet(VAL_SCALED_PATH)
test_scaled_df = pd.read_parquet(TEST_SCALED_PATH)
final_feature_columns_used = joblib.load(FEATURE_LIST_FILENAME)

enkf_train_df = pd.read_parquet(ENKF_TRAIN_DF_PATH)
enkf_validation_df = pd.read_parquet(ENKF_VAL_DF_PATH)
enkf_test_df = pd.read_parquet(ENKF_TEST_DF_PATH)
enkf_train_scaled_df = pd.read_parquet(ENKF_TRAIN_SCALED_PATH)
enkf_validation_scaled_df = pd.read_parquet(ENKF_VAL_SCALED_PATH)
enkf_test_scaled_df = pd.read_parquet(ENKF_TEST_SCALED_PATH)
enkf_feature_columns_used = joblib.load(ENKF_FEATURE_LIST_FILENAME)

def _make_tuning_df(train_scaled, val_scaled, feature_cols):
    cols = list(feature_cols)
    if TARGET_BINARY_COL not in cols:
        cols.append(TARGET_BINARY_COL)
    return (
        pd.concat([train_scaled[cols], val_scaled[cols]], axis=0).sort_index(),
        cols.index(TARGET_BINARY_COL),
        cols,
    )

TUNING_DATASETS = {
    "raw": {
        "label": "Standalone LSTM",
        "train_scaled": train_scaled_df,
        "val_scaled": validation_scaled_df,
        "test_scaled": test_scaled_df,
        "feature_columns": final_feature_columns_used,
        "tuned_sequences_path": RAW_TUNED_SEQUENCES_PATH,
        "hps_path": RAW_BEST_HPS_PATH,
    },
    "enkf": {
        "label": "LSTM-EnKF preprocessor",
        "train_scaled": enkf_train_scaled_df,
        "val_scaled": enkf_validation_scaled_df,
        "test_scaled": enkf_test_scaled_df,
        "feature_columns": enkf_feature_columns_used,
        "tuned_sequences_path": ENKF_TUNED_SEQUENCES_PATH,
        "hps_path": ENKF_BEST_HPS_PATH,
    },
}

for dataset_name, ds in TUNING_DATASETS.items():
    tuning_df_i, target_idx_i, cols_i = _make_tuning_df(
        ds["train_scaled"], ds["val_scaled"], ds["feature_columns"]
    )
    ds["tuning_df"] = tuning_df_i
    ds["target_idx"] = target_idx_i
    ds["cols_to_use"] = cols_i
    print(f"{dataset_name:>4s}: tuning_df={tuning_df_i.shape}, target_idx={target_idx_i}, features={len(cols_i)}")

# Backward-compatible aliases for existing diagnostic cells.
tuning_df = TUNING_DATASETS["raw"]["tuning_df"]
TARGET_IDX_IN_FEATURES = TUNING_DATASETS["raw"]["target_idx"]
feature_columns = TUNING_DATASETS["raw"]["cols_to_use"]


 raw: tuning_df=(1363, 53), target_idx=52, features=53
enkf: tuning_df=(1363, 59), target_idx=58, features=59


In [7]:
# Sanity check
print('Class weight examples for current train target:')
y_demo = train_df[TARGET_BINARY_COL].values
for p in [0.0, 0.5, 1.0, 1.5]:
    print(f'  power={p}: {class_weights_powered(y_demo, p)}')


Class weight examples for current train target:
  power=0.0: {0: 1.0, 1: 1.0}
  power=0.5: {0: 0.8066209219350564, 1: 1.4695629910335197}
  power=1.0: {0: 0.6506373117033604, 1: 2.1596153846153845}
  power=1.5: {0: 0.5248176682115112, 1: 3.1736908440973894}


In [8]:
def sample_hps(trial):
    """Sample one hyperparameter configuration from the search space."""
    hps = {
        # Data shape
        'seq_length':        trial.suggest_categorical('seq_length', [4, 8, 12, 16, 26]),
        # Architecture
        'n_lstm_layers':     trial.suggest_int('n_lstm_layers', 1, 3),
        'units_1':           trial.suggest_categorical('units_1', [32, 48, 64, 96, 128]),
        'units_2':           trial.suggest_categorical('units_2', [16, 32, 48, 64]),
        'units_3':           trial.suggest_categorical('units_3', [8, 16, 32]),
        # Regularization
        'dropout':           trial.suggest_float('dropout', 0.1, 0.5),
        'recurrent_dropout': trial.suggest_float('recurrent_dropout', 0.0, 0.4),
        'l2_reg':            trial.suggest_float('l2_reg', 1e-7, 1e-2, log=True),
        # Optimizer
        'optimizer':         trial.suggest_categorical('optimizer', ['adam', 'adamw', 'nadam']),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True),
        'weight_decay':      trial.suggest_float('weight_decay', 1e-7, 1e-2, log=True),
        # Training
        'batch_size':        trial.suggest_categorical('batch_size', [16, 32, 64, 128]),
        # Imbalance handling
        'cw_power':          trial.suggest_float('cw_power', 0.0, 1.5),
    }
    return hps


In [9]:
from sklearn.model_selection import TimeSeriesSplit
from optuna_integration import TFKerasPruningCallback


def make_objective(dataset_name, tuning_df_i, target_idx_i):
    """Original AUPRC objective. Optimizes mean validation AUPRC under
    class weighting, penalized by fold-to-fold instability."""

    def objective(trial):
        hps = sample_hps(trial)
        try:
            X_full, y_full = create_sequences_from_df(
                tuning_df_i,
                seq_length=hps["seq_length"],
                target_col_idx=target_idx_i,
                pred_step=FORECAST_HORIZON,
                return_targets=True,
            )
        except Exception as e:
            raise optuna.TrialPruned(f"{dataset_name}: sequence creation failed: {e}")

        if len(X_full) < N_CV_FOLDS * 20:
            raise optuna.TrialPruned(f"{dataset_name}: too few sequences ({len(X_full)})")

        X_full = np.asarray(X_full, dtype=np.float32)
        y_full = np.asarray(y_full, dtype=np.float32).flatten()
        test_size = max(20, len(y_full) // (N_CV_FOLDS + 1))
        tscv = TimeSeriesSplit(n_splits=N_CV_FOLDS, test_size=test_size)

        fold_scores = []
        for fold_idx, (tr_idx, val_idx) in enumerate(tscv.split(X_full)):
            if len(np.unique(y_full[tr_idx])) < 2 or len(np.unique(y_full[val_idx])) < 2:
                continue

            tf.keras.backend.clear_session()
            model = build_tunable_lstm((hps["seq_length"], X_full.shape[2]), hps)
            cw = class_weights_powered(y_full[tr_idx], power=hps["cw_power"])
            callbacks = [
                EarlyStopping(monitor="val_auprc", mode="max",
                              patience=EARLY_STOP_PAT, restore_best_weights=True)
            ]
            if fold_idx == 0:
                callbacks.append(TFKerasPruningCallback(trial, "val_auprc"))

            try:
                model.fit(
                    X_full[tr_idx], y_full[tr_idx],
                    validation_data=(X_full[val_idx], y_full[val_idx]),
                    epochs=MAX_EPOCHS,
                    batch_size=hps["batch_size"],
                    class_weight=cw,
                    callbacks=callbacks,
                    verbose=0,
                )
            except optuna.TrialPruned:
                raise
            except Exception as e:
                raise optuna.TrialPruned(
                    f"{dataset_name}: training crashed in fold {fold_idx}: {e}"
                )

            val_pred = model.predict(X_full[val_idx], verbose=0).flatten()
            if np.isnan(val_pred).any():
                raise optuna.TrialPruned(f"{dataset_name}: NaN predictions")
            fold_scores.append(average_precision_score(y_full[val_idx], val_pred))
            trial.report(float(np.mean(fold_scores)), step=fold_idx + 1)
            if trial.should_prune():
                raise optuna.TrialPruned(f"{dataset_name}: pruned after fold {fold_idx}")

        if not fold_scores:
            raise optuna.TrialPruned(f"{dataset_name}: no usable folds")

        mean_score = float(np.mean(fold_scores))
        std_score = float(np.std(fold_scores))
        trial.set_user_attr("dataset", dataset_name)
        trial.set_user_attr("fold_auprcs", fold_scores)
        trial.set_user_attr("mean_auprc", mean_score)
        trial.set_user_attr("std_auprc", std_score)
        trial.set_user_attr("n_folds_used", len(fold_scores))
        return mean_score - STABILITY_LAMBDA * std_score

    return objective


In [10]:
# Onset-CSI tuning objective - sibling to make_objective above.
# Optimizes event-based onset CSI on each TimeSeriesSplit fold under
# onset-aware sample weighting.

import onset_utils as ou

ONSET_BUFFER_WEEKS = 4
ONSET_EVENT_WINDOW = 2
ONSET_MULTIPLIER = 5.0
PRE_ONSET_MULTIPLIER = 3.0
PRE_ONSET_WINDOW = 4


def make_objective_onset(dataset_name, tuning_df_i, target_idx_i):
    """Optuna objective that maximizes event-based onset CSI under
    onset-aware sample weighting. Same signature as make_objective."""

    def objective(trial):
        hps = sample_hps(trial)
        try:
            X_full, y_full = create_sequences_from_df(
                tuning_df_i,
                seq_length=hps["seq_length"],
                target_col_idx=target_idx_i,
                pred_step=FORECAST_HORIZON,
                return_targets=True,
            )
        except Exception as e:
            raise optuna.TrialPruned(f"{dataset_name}: sequence creation failed: {e}")

        if len(X_full) < N_CV_FOLDS * 20:
            raise optuna.TrialPruned(f"{dataset_name}: too few sequences ({len(X_full)})")

        X_full = np.asarray(X_full, dtype=np.float32)
        y_full = np.asarray(y_full, dtype=np.float32).flatten()
        test_size = max(20, len(y_full) // (N_CV_FOLDS + 1))
        tscv = TimeSeriesSplit(n_splits=N_CV_FOLDS, test_size=test_size)

        fold_scores = []
        for fold_idx, (tr_idx, val_idx) in enumerate(tscv.split(X_full)):
            if len(np.unique(y_full[tr_idx])) < 2 or len(np.unique(y_full[val_idx])) < 2:
                continue

            tf.keras.backend.clear_session()
            model = build_tunable_lstm((hps["seq_length"], X_full.shape[2]), hps)
            cw = class_weights_powered(y_full[tr_idx], power=hps["cw_power"])
            sample_w = ou.onset_sample_weights(
                y_full[tr_idx],
                onset_multiplier=ONSET_MULTIPLIER,
                pre_onset_multiplier=PRE_ONSET_MULTIPLIER,
                pre_onset_window=PRE_ONSET_WINDOW,
                buffer_weeks=ONSET_BUFFER_WEEKS,
                class_weight=cw,
            )

            callbacks = [
                EarlyStopping(monitor="val_auprc", mode="max",
                              patience=EARLY_STOP_PAT, restore_best_weights=True)
            ]
            if fold_idx == 0:
                callbacks.append(TFKerasPruningCallback(trial, "val_auprc"))

            try:
                model.fit(
                    X_full[tr_idx], y_full[tr_idx],
                    validation_data=(X_full[val_idx], y_full[val_idx]),
                    epochs=MAX_EPOCHS,
                    batch_size=hps["batch_size"],
                    sample_weight=sample_w,
                    callbacks=callbacks,
                    verbose=0,
                )
            except optuna.TrialPruned:
                raise
            except Exception as e:
                raise optuna.TrialPruned(
                    f"{dataset_name}: training crashed in fold {fold_idx}: {e}"
                )

            val_pred = model.predict(X_full[val_idx], verbose=0).flatten()
            if np.isnan(val_pred).any():
                raise optuna.TrialPruned(f"{dataset_name}: NaN predictions")

            # Onset CSI on this fold at the CSI-optimal threshold.
            _, metrics = ou.best_onset_csi_threshold(
                val_pred,
                y_full[val_idx],
                buffer_weeks=ONSET_BUFFER_WEEKS,
                window=ONSET_EVENT_WINDOW,
            )
            fold_scores.append(metrics.get("CSI", 0.0))
            trial.report(float(np.mean(fold_scores)), step=fold_idx + 1)
            if trial.should_prune():
                raise optuna.TrialPruned(f"{dataset_name}: pruned after fold {fold_idx}")

        if not fold_scores:
            raise optuna.TrialPruned(f"{dataset_name}: no usable folds")

        mean_score = float(np.mean(fold_scores))
        std_score = float(np.std(fold_scores))
        trial.set_user_attr("dataset", dataset_name)
        trial.set_user_attr("fold_onset_csis", fold_scores)
        trial.set_user_attr("mean_onset_csi", mean_score)
        trial.set_user_attr("std_onset_csi", std_score)
        trial.set_user_attr("n_folds_used", len(fold_scores))
        return mean_score - STABILITY_LAMBDA * std_score

    return objective


## Run the Study

Trials run sequentially by default. If you have multiple GPUs, you can launch parallel workers pointing at the same SQLite store. The study auto-resumes from `STORAGE_PATH` on rerun.


In [11]:
def run_study_for_dataset(dataset_name, ds, objective_factory=None, study_suffix=""):
    """Create/load an Optuna study and optimize it.

    objective_factory: callable (dataset_name, tuning_df, target_idx) -> objective.
                       Defaults to make_objective (AUPRC).
    study_suffix:      appended to STUDY_NAME so onset studies don't collide
                       with the AUPRC studies in the SQLite storage.
    """
    if objective_factory is None:
        objective_factory = make_objective

    study_name = f"{STUDY_NAME}{study_suffix}_{dataset_name}"
    storage_path = f"sqlite:///{OUTPUT_DIR}/optuna_{study_name}.db"
    study = optuna.create_study(
        study_name=study_name,
        storage=storage_path,
        load_if_exists=True,
        direction="maximize",
        sampler=TPESampler(
            n_startup_trials=N_STARTUP_TRIALS,
            multivariate=True,
            group=True,
            seed=42,
        ),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=10 if not FAST_TEST else 1,
            n_warmup_steps=15 if not FAST_TEST else 2,
            interval_steps=5,
        ),
    )

    existing_trials = len(study.trials)
    print(f"\n=== {ds['label']} ({dataset_name}{study_suffix}) ===")
    print(f"Study: {study_name}")
    print(f"Storage: {storage_path}")
    print(f"Existing trials: {existing_trials}")

    if existing_trials == 0:
        study.enqueue_trial(dict(DEFAULT_HPS_DICT))
        print("Enqueued DEFAULT_HPS_DICT as warm start.")

    objective = objective_factory(dataset_name, ds["tuning_df"], ds["target_idx"])
    study.optimize(
        objective,
        n_trials=max(0, N_TRIALS - existing_trials),
        timeout=MAX_TIME_HOURS * 3600,
        show_progress_bar=True,
        gc_after_trial=True,
    )
    complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    failed = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
    print(f"Done: total={len(study.trials)}, complete={complete}, pruned={pruned}, failed={failed}")
    return study


# --- Run the AUPRC studies ---------------------------------------------------
studies = {}
if PERFORM_TUNING:
    for dataset_name, ds in TUNING_DATASETS.items():
        studies[dataset_name] = run_study_for_dataset(
            dataset_name, ds,
            objective_factory=make_objective,
            study_suffix="",
        )
else:
    print("PERFORM_TUNING=False; downstream notebooks will use DEFAULT_HPS_DICT.")

# Backward-compatible alias used by the plotting cells below.
study = studies.get("raw")


# --- Run the onset-CSI studies (optional) -----------------------------------
# Set PERFORM_ONSET_TUNING=True at the top of the notebook to enable.
PERFORM_ONSET_TUNING = globals().get("PERFORM_ONSET_TUNING", True)

onset_studies = {}
if PERFORM_ONSET_TUNING and PERFORM_TUNING:
    for dataset_name, ds in TUNING_DATASETS.items():
        onset_studies[dataset_name] = run_study_for_dataset(
            dataset_name, ds,
            objective_factory=make_objective_onset,
            study_suffix="_onset",
        )
else:
    if PERFORM_TUNING:
        print("\nPERFORM_ONSET_TUNING=False; onset studies skipped.")


C:\Users\victo\AppData\Local\Temp\ipykernel_39840\3418378154.py:19: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=TPESampler(
C:\Users\victo\AppData\Local\Temp\ipykernel_39840\3418378154.py:19: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler=TPESampler(
[I 2026-06-15 00:51:32,496] A new study created in RDB with name: hab_lstm_v3_raw



=== Standalone LSTM (raw) ===
Study: hab_lstm_v3_raw
Storage: sqlite:///output_refactored//optuna_hab_lstm_v3_raw.db
Existing trials: 0
Enqueued DEFAULT_HPS_DICT as warm start.


  0%|          | 0/60 [00:00<?, ?it/s]

C:\Users\victo\AppData\Local\Temp\ipykernel_39840\4096175687.py:66: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  trial.report(float(np.mean(fold_scores)), step=fold_idx + 1)
C:\Users\victo\AppData\Local\Temp\ipykernel_39840\4096175687.py:66: UserWarning: The reported value is ignored because this `step` 2 is already reported.
  trial.report(float(np.mean(fold_scores)), step=fold_idx + 1)
C:\Users\victo\AppData\Local\Temp\ipykernel_39840\4096175687.py:66: UserWarning: The reported value is ignored because this `step` 3 is already reported.
  trial.report(float(np.mean(fold_scores)), step=fold_idx + 1)
Best trial: 0. Best value: 0.591433:   2%|▏         | 1/60 [00:11<11:31, 11.72s/it, 11.72/28800 seconds]

[I 2026-06-15 00:51:44,065] Trial 0 finished with value: 0.5914331791179845 and parameters: {'seq_length': 26, 'n_lstm_layers': 2, 'units_1': 64, 'units_2': 32, 'units_3': 16, 'dropout': 0.3, 'recurrent_dropout': 0.0, 'l2_reg': 1e-06, 'optimizer': 'adam', 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 32, 'cw_power': 1.0}. Best is trial 0 with value: 0.5914331791179845.


...

[I 2026-06-15 02:55:13,941] Trial 58 finished with value: 0.08322290602219082 and parameters: {'seq_length': 26, 'n_lstm_layers': 1, 'units_1': 32, 'units_2': 32, 'units_3': 16, 'dropout': 0.23120638441285749, 'recurrent_dropout': 0.16060275867876067, 'l2_reg': 1.2186903808752616e-07, 'optimizer': 'adam', 'learning_rate': 3.6878756527718056e-05, 'weight_decay': 0.004775196473331208, 'batch_size': 64, 'cw_power': 0.2579094796696612}. Best is trial 51 with value: 0.1427080358612573.


Best trial: 51. Best value: 0.142708:  98%|█████████▊| 59/60 [45:28<00:46, 46.16s/it, 2679.10/28800 seconds]

[I 2026-06-15 02:56:12,399] Trial 59 finished with value: 0.11126942891612224 and parameters: {'seq_length': 8, 'n_lstm_layers': 1, 'units_1': 128, 'units_2': 48, 'units_3': 16, 'dropout': 0.2791987848846535, 'recurrent_dropout': 0.3592513548727783, 'l2_reg': 6.993469266175501e-07, 'optimizer': 'adamw', 'learning_rate': 1.1942275520817121e-05, 'weight_decay': 0.006385797089589973, 'batch_size': 64, 'cw_power': 0.3387598330066678}. Best is trial 51 with value: 0.1427080358612573.


Best trial: 51. Best value: 0.142708: 100%|██████████| 60/60 [45:37<00:00, 45.63s/it, 2737.57/28800 seconds]

Done: total=60, complete=41, pruned=19, failed=0


## Extract `best_hps` for Part 10

The downstream `train_final_model` function in Part 10 expects a `best_hps` object with a `.get()` method (KerasTuner's `HyperParameters` instance). We construct a lightweight shim that exposes the same interface, so Part 10 needs **no changes**.

We also surface the best `seq_length` separately — if it differs from `SEQUENCE_LENGTH`, you'll need to rebuild `X_train/val/test` before Part 10.


In [12]:
def _trial_score(t):
    ma = t.user_attrs.get("mean_auprc")
    if ma is not None:
        return float(ma)
    vals = list(t.intermediate_values.values()) if t.intermediate_values else []
    return max(vals) if vals else float("-inf")


def select_best_hps(study_i):
    if study_i is None:
        return dict(DEFAULT_HPS_DICT), None, float("nan"), "DEFAULTS"

    completed = [t for t in study_i.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if completed:
        best_trial_i = study_i.best_trial
        return dict(best_trial_i.params), best_trial_i, float(best_trial_i.value), "COMPLETE"

    scored = [(t, _trial_score(t)) for t in study_i.trials
              if t.state == optuna.trial.TrialState.PRUNED]
    scored = [(t, s) for t, s in scored if s > float("-inf")]
    if scored:
        best_trial_i, best_value_i = max(scored, key=lambda ts: ts[1])
        print(f"WARNING: {study_i.study_name} had no completed trials; using best pruned trial.")
        return dict(best_trial_i.params), best_trial_i, float(best_value_i), "PRUNED"

    print(f"WARNING: {study_i.study_name} had no usable trials; using DEFAULT_HPS_DICT.")
    return dict(DEFAULT_HPS_DICT), None, float("nan"), "DEFAULTS"


# Best HPs from the AUPRC studies.
best_hps_by_dataset = {}
for dataset_name, ds in TUNING_DATASETS.items():
    params, trial, value, src = select_best_hps(studies.get(dataset_name))
    params = {**DEFAULT_HPS_DICT, **params}
    best_hps_by_dataset[dataset_name] = {
        "hps": params,
        "trial_number": None if trial is None else int(trial.number),
        "objective_value": value,
        "source": src,
        "mean_auprc": None if trial is None else trial.user_attrs.get("mean_auprc"),
        "std_auprc": None if trial is None else trial.user_attrs.get("std_auprc"),
        "n_trials_completed": 0 if studies.get(dataset_name) is None else len([
            t for t in studies[dataset_name].trials if t.state == optuna.trial.TrialState.COMPLETE
        ]),
    }
    print(f"\n=== Best HPs (AUPRC): {ds['label']} ({src}) ===")
    print(f"Objective: {value if value == value else 'N/A'}")
    for k, v in params.items():
        print(f"  {k:<20s} = {v}")


# Best HPs from the onset-CSI studies (only if those ran).
best_hps_by_dataset_onset = {}
for dataset_name, ds in TUNING_DATASETS.items():
    st = onset_studies.get(dataset_name)
    if st is None:
        continue
    params, trial, value, src = select_best_hps(st)
    params = {**DEFAULT_HPS_DICT, **params}
    best_hps_by_dataset_onset[dataset_name] = {
        "hps": params,
        "trial_number": None if trial is None else int(trial.number),
        "objective_value": value,
        "source": src,
        "mean_onset_csi": None if trial is None else trial.user_attrs.get("mean_onset_csi"),
        "std_onset_csi":  None if trial is None else trial.user_attrs.get("std_onset_csi"),
        "n_trials_completed": len([
            t for t in st.trials if t.state == optuna.trial.TrialState.COMPLETE
        ]),
    }
    print(f"\n=== Best HPs (Onset CSI): {ds['label']} ({src}) ===")
    print(f"Objective: {value if value == value else 'N/A'}")
    for k, v in params.items():
        print(f"  {k:<20s} = {v}")


# Backward-compatible aliases for downstream diagnostics.
best_params = best_hps_by_dataset["raw"]["hps"]
best_hps = HPsShim(best_params)
best_trial = None
if studies.get("raw") is not None:
    completed_raw = [t for t in studies["raw"].trials if t.state == optuna.trial.TrialState.COMPLETE]
    best_trial = studies["raw"].best_trial if completed_raw else None
BEST_SEQ_LENGTH = best_params["seq_length"]
BEST_BATCH_SIZE = best_params["batch_size"]



=== Best HPs (AUPRC): Standalone LSTM (COMPLETE) ===
Objective: 0.7539556748900048
  seq_length           = 26
  n_lstm_layers        = 1
  units_1              = 128
  units_2              = 48
  units_3              = 16
  dropout              = 0.2730850543622167
  recurrent_dropout    = 0.06848901466850234
  l2_reg               = 0.005804918908645152
  optimizer            = adam
  learning_rate        = 0.009663137093986352
  weight_decay         = 9.521725683987584e-05
  batch_size           = 16
  cw_power             = 1.4834602070155125

=== Best HPs (AUPRC): LSTM-EnKF preprocessor (COMPLETE) ===
Objective: 0.7227269986707757
  seq_length           = 26
  n_lstm_layers        = 1
  units_1              = 128
  units_2              = 32
  units_3              = 8
  dropout              = 0.17838595187080625
  recurrent_dropout    = 0.048548763366217114
  l2_reg               = 0.0003102673360051331
  optimizer            = adam
  learning_rate        = 0.009940236532651029
  

In [13]:
# Persist tuned sequence artifacts. Notebook 03 loads these directly, so
# notebook 02 no longer edits shared_config.py or requires notebook 01 reruns
# when Optuna chooses a different sequence length.
for dataset_name, ds in TUNING_DATASETS.items():
    hps_i = best_hps_by_dataset[dataset_name]["hps"]
    seq_len_i = int(hps_i["seq_length"])
    sequences_i = build_sequence_artifact(
        ds["train_scaled"],
        ds["val_scaled"],
        ds["test_scaled"],
        ds["feature_columns"],
        TARGET_BINARY_COL,
        seq_len_i,
        FORECAST_HORIZON,
        ds["tuned_sequences_path"],
    )
    print(
        f"{dataset_name:>4s}: wrote {ds['tuned_sequences_path']} "
        f"(seq_length={seq_len_i}, X_train={sequences_i['X_train'].shape})"
    )

# Backward-compatible in-kernel raw arrays for the remaining diagnostic cells.
raw_tuned = load_sequence_artifact(RAW_TUNED_SEQUENCES_PATH)
X_train = raw_tuned["X_train"]; y_train = raw_tuned["y_train"]
X_val = raw_tuned["X_val"]; y_val = raw_tuned["y_val"]
X_test = raw_tuned["X_test"]; y_test = raw_tuned["y_test"]
feature_columns = raw_tuned["feature_columns"]
SEQUENCE_LENGTH = raw_tuned["seq_length"]
print(f"Raw tuned arrays now active in kernel: X_train={X_train.shape}, seq_length={SEQUENCE_LENGTH}")


 raw: wrote output_refactored/intermediate\sequences_lstm_best.npz (seq_length=26, X_train=(1097, 26, 52))
enkf: wrote output_refactored/intermediate\sequences_lstm_enkf_best.npz (seq_length=26, X_train=(1097, 26, 58))
Raw tuned arrays now active in kernel: X_train=(1097, 26, 52), seq_length=26


## Diagnostics: What Did the Tuner Learn?

These plots are publication-grade supplementary material. They tell a story:
- **Importance** — which hyperparameters actually matter
- **Parallel coordinates** — visualize the high-performing region of the search space
- **Optimization history** — did the tuner converge, or is more budget needed?
- **Slice plot** — marginal effect of each hyperparameter on the objective


In [14]:
import optuna.visualization as ovis

def _save_plotly(fig, fname):
    """Save fig to PNG (or HTML fallback) and try to render inline.

    Inline rendering needs nbformat>=4.2.0 in the venv. If that's missing
    (or any other renderer issue happens), we just log it and move on so
    the rest of the cell still runs.
    """
    path = os.path.join(RESULTS_DIR, fname)
    try:
        fig.write_image(path, scale=2)
        print(f'Saved: {path}')
    except Exception as e:
        print(f'PNG export failed ({e}); saving HTML instead.')
        fig.write_html(path.replace('.png', '.html'))
    try:
        fig.show()
    except Exception as e:
        print(f'  (inline render skipped: {e})')

# 1. Hyperparameter importance (uses fANOVA - needs >=2 completed trials)
try:
    fig_imp = ovis.plot_param_importances(study)
    fig_imp.update_layout(title='Hyperparameter Importance', width=900, height=500)
    _save_plotly(fig_imp, 'param_importance.png')
except Exception as e:
    print(f'Importance plot skipped: {e}')

# 2. Optimization history
try:
    fig_hist = ovis.plot_optimization_history(study)
    fig_hist.update_layout(title='Optimization History', width=900, height=450)
    _save_plotly(fig_hist, 'optimization_history.png')
except Exception as e:
    print(f'Optimization history plot skipped: {e}')

# 3. Parallel coordinates over top-performing trials
try:
    fig_par = ovis.plot_parallel_coordinate(study)
    fig_par.update_layout(title='Parallel Coordinates - All Trials',
                          width=1100, height=550)
    _save_plotly(fig_par, 'parallel_coordinates.png')
except Exception as e:
    print(f'Parallel coordinates plot skipped: {e}')

# 4. Slice plot for the most impactful continuous params
try:
    fig_slice = ovis.plot_slice(study, params=['learning_rate', 'dropout',
                                                'recurrent_dropout', 'l2_reg'])
    fig_slice.update_layout(title='Marginal Effects on Objective',
                            width=1100, height=400)
    _save_plotly(fig_slice, 'slice_plot.png')
except Exception as e:
    print(f'Slice plot skipped: {e}')


Saved: output_refactored/tuning_results\param_importance.png
  (inline render skipped: Mime type rendering requires nbformat>=4.2.0 but it is not installed)
Saved: output_refactored/tuning_results\optimization_history.png
  (inline render skipped: Mime type rendering requires nbformat>=4.2.0 but it is not installed)
Saved: output_refactored/tuning_results\parallel_coordinates.png
  (inline render skipped: Mime type rendering requires nbformat>=4.2.0 but it is not installed)
Saved: output_refactored/tuning_results\slice_plot.png
  (inline render skipped: Mime type rendering requires nbformat>=4.2.0 but it is not installed)


In [15]:
if study is None:
    print('No Optuna study object is active (PERFORM_TUNING=False); skipping trial table.')
    df_trials = pd.DataFrame()
else:
    # 5. Top-N trials table (for the paper's supplementary)
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned    = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    failed    = [t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]
    print(f'Trial state breakdown: completed={len(completed)}, '
          f'pruned={len(pruned)}, failed={len(failed)}')

    # If no trial completed, fall back to ranking pruned trials by their best
    # recorded intermediate / user_attr score so the table is still informative.
    if completed:
        rows = [{
            'trial':      t.number,
            'state':      'COMPLETE',
            'objective':  t.value,
            'mean_auprc': t.user_attrs.get('mean_auprc', np.nan),
            'std_auprc':  t.user_attrs.get('std_auprc',  np.nan),
            **t.params,
        } for t in completed]
        sort_col = 'objective'
    elif pruned:
        print('*** No completed trials. Ranking PRUNED trials by best recorded score. '
              'Delete the SQLite study DB and rerun Cell 32 to get real results.')
        def _score(t):
            ma = t.user_attrs.get('mean_auprc')
            if ma is not None:
                return float(ma)
            vals = list(t.intermediate_values.values()) if t.intermediate_values else []
            return float(max(vals)) if vals else float('nan')
        rows = [{
            'trial':       t.number,
            'state':       'PRUNED',
            'best_score':  _score(t),
            'mean_auprc':  t.user_attrs.get('mean_auprc', np.nan),
            **t.params,
        } for t in pruned]
        sort_col = 'best_score'
    else:
        print('*** No completed or pruned trials -- nothing to rank.')
        rows = []
        sort_col = None

    df_trials = pd.DataFrame(rows)
    if sort_col is not None and not df_trials.empty:
        df_trials = (df_trials.sort_values(sort_col, ascending=False, na_position='last')
                              .reset_index(drop=True))

    print('\nTop 10 trials:')
    print(df_trials.head(10).to_string(index=False) if not df_trials.empty
          else '(empty - study has no usable trials)')

    csv_path = os.path.join(RESULTS_DIR, 'all_trials.csv')
    df_trials.to_csv(csv_path, index=False)
    print(f'\nFull trials log saved: {csv_path}')


Trial state breakdown: completed=57, pruned=3, failed=0

Top 10 trials:
 trial    state  objective  mean_auprc  std_auprc  seq_length  n_lstm_layers  units_1  units_2  units_3  dropout  recurrent_dropout   l2_reg optimizer  learning_rate  weight_decay  batch_size  cw_power
    54 COMPLETE   0.753956    0.780278   0.087742          26              1      128       48       16 0.273085           0.068489 0.005805      adam       0.009663      0.000095          16  1.483460
    50 COMPLETE   0.744948    0.775370   0.101406          26              1       64       48       16 0.279696           0.073169 0.000004     nadam       0.006671      0.000539          16  1.430257
    59 COMPLETE   0.742638    0.770800   0.093874          26              1      128       16       16 0.405823           0.038282 0.000045     nadam       0.007363      0.000127          16  1.010016
    57 COMPLETE   0.742264    0.770762   0.094991          26              1      128       16       16 0.357043        

## Bonus: Post-Hoc Decision Threshold Tuning

Don't evaluate the final model at threshold = 0.5. For HAB warnings, missing a bloom usually costs more than a false alarm, so we tune for $F_\beta$ with $\beta = 2$ (recall-weighted). The chosen threshold gets passed into Part 11 / 15.

We run this after Part 10 finishes training — but defining it here keeps all the tuning logic in one place.


In [16]:
def tune_decision_threshold(model, X_val_arr, y_val_arr, beta=2.0,
                            search_space=None, plot=True):
    """
    Sweep thresholds on validation, return the one maximizing F-beta.
    Falls back to 0.5 if no positive class is present.
    """
    y_val_arr = np.asarray(y_val_arr).flatten()
    if len(np.unique(y_val_arr)) < 2:
        print('Warning: validation has only one class. Returning 0.5.')
        return 0.5, None

    val_pred = model.predict(X_val_arr, verbose=0).flatten()
    precision, recall, thresholds = precision_recall_curve(y_val_arr, val_pred)
    # precision_recall_curve returns one extra precision/recall point; trim
    precision, recall = precision[:-1], recall[:-1]
    f_beta = ((1 + beta**2) * precision * recall /
              (beta**2 * precision + recall + 1e-12))
    best_idx = int(np.argmax(f_beta))
    best_thresh = float(thresholds[best_idx])

    if plot:
        fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
        ax[0].plot(thresholds, precision, label='Precision', color='#1f77b4')
        ax[0].plot(thresholds, recall,    label='Recall',    color='#d62728')
        ax[0].plot(thresholds, f_beta,    label=f'F{beta:g}',color='#2ca02c', lw=2)
        ax[0].axvline(best_thresh, ls='--', color='black',
                      label=f'best={best_thresh:.3f}')
        ax[0].set_xlabel('Threshold'); ax[0].set_ylabel('Score')
        ax[0].set_title(f'Threshold sweep (F{beta:g} optimal)')
        ax[0].legend(); ax[0].grid(alpha=0.3)

        ax[1].plot(recall, precision, color='purple', lw=2)
        ax[1].scatter([recall[best_idx]], [precision[best_idx]],
                      color='red', s=80, zorder=5,
                      label=f'F{beta:g}-optimal')
        ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
        ax[1].set_title('Precision-Recall curve'); ax[1].legend(); ax[1].grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, 'threshold_tuning.png'),
                    dpi=200, bbox_inches='tight')
        plt.show()

    return best_thresh, {
        'precision': precision[best_idx],
        'recall':    recall[best_idx],
        'f_beta':    f_beta[best_idx],
    }

# Usage (call AFTER Part 10 trains the final model):
#   BEST_THRESHOLD, threshold_metrics = tune_decision_threshold(
#       trained_model, X_val, y_val, beta=2.0)
#   print(f'Use BEST_THRESHOLD = {BEST_THRESHOLD:.3f} in Part 11 / 15.')
print('Function tune_decision_threshold() defined. Call after Part 10.')


Function tune_decision_threshold() defined. Call after Part 10.


In [17]:
# Persist a compact tuning manifest for review/reproducibility.
tuning_manifest = {}
for dataset_name, ds in TUNING_DATASETS.items():
    study_i = studies.get(dataset_name)
    tuning_manifest[dataset_name] = {
        "label": ds["label"],
        "study_name": None if study_i is None else study_i.study_name,
        "n_trials_total": 0 if study_i is None else len(study_i.trials),
        **best_hps_by_dataset[dataset_name],
        "tuned_sequences_path": ds["tuned_sequences_path"],
    }

write_json(os.path.join(INTERMEDIATE_DIR, "tuning_manifest.json"), tuning_manifest)
print(json.dumps(tuning_manifest, indent=2, default=str))


{
  "raw": {
    "label": "Standalone LSTM",
    "study_name": "hab_lstm_v3_raw",
    "n_trials_total": 60,
    "hps": {
      "seq_length": 26,
      "n_lstm_layers": 1,
      "units_1": 128,
      "units_2": 48,
      "units_3": 16,
      "dropout": 0.2730850543622167,
      "recurrent_dropout": 0.06848901466850234,
      "l2_reg": 0.005804918908645152,
      "optimizer": "adam",
      "learning_rate": 0.009663137093986352,
      "weight_decay": 9.521725683987584e-05,
      "batch_size": 16,
      "cw_power": 1.4834602070155125
    },
    "trial_number": 54,
    "objective_value": 0.7539556748900048,
    "source": "COMPLETE",
    "mean_auprc": 0.7802783770164156,
    "std_auprc": 0.08774234042136926,
    "n_trials_completed": 57,
    "tuned_sequences_path": "output_refactored/intermediate\\sequences_lstm_best.npz"
  },
  "enkf": {
    "label": "LSTM-EnKF preprocessor",
    "study_name": "hab_lstm_v3_enkf",
    "n_trials_total": 60,
    "hps": {
      "seq_length": 26,
      "n_lstm_l

## Persist `best_hps` for downstream notebooks

Notebook 03 (training) reads this JSON instead of re-running the Optuna study.

In [18]:
# Persist AUPRC-tuned hyperparameters for downstream notebooks.
for dataset_name, ds in TUNING_DATASETS.items():
    payload = best_hps_by_dataset[dataset_name]["hps"]
    write_json(ds["hps_path"], payload)
    print(f"Wrote {ds['hps_path']}")

# Backward-compatible raw alias used by older cells/scripts.
write_json(BEST_HPS_PATH, best_hps_by_dataset["raw"]["hps"])
print(f"Wrote raw alias {BEST_HPS_PATH}")

# Persist onset-tuned hyperparameters under separate filenames so they don't
# collide with the AUPRC HPs.
for dataset_name, bundle in best_hps_by_dataset_onset.items():
    tag = "lstm" if dataset_name == "raw" else "lstm_enkf"
    out_path = os.path.join(INTERMEDIATE_DIR, f"best_hps_{tag}_onset.json")
    write_json(out_path, bundle["hps"])
    print(f"Wrote {out_path} (onset-tuned)")


Wrote output_refactored/intermediate\best_hps_lstm.json
Wrote output_refactored/intermediate\best_hps_lstm_enkf.json
Wrote raw alias output_refactored/intermediate\best_hps.json
Wrote output_refactored/intermediate\best_hps_lstm_onset.json (onset-tuned)
Wrote output_refactored/intermediate\best_hps_lstm_enkf_onset.json (onset-tuned)
